In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [10]:
# coding a kernel SVM.
data=pd.read_csv('dataset/unlinear_data.csv')
label=pd.read_csv('dataset/unlinear_label.csv')
## trans to numpy array
X=data.values
y=label.values
print(X.shape,y.shape)

(500, 2) (500, 1)


In [ ]:
## kernel slection
## 1. guassian
## 2. polynomial
## use gaussian as example
def gaussian_kernel(X1, X2, sigma):
    
    X1_sq = np.sum(X1**2, axis=1).reshape(-1, 1)
    X2_sq = np.sum(X2**2, axis=1).reshape(1, -1)
    
    dists_sq = X1_sq + X2_sq - 2 * X1 @ X2.T
    return np.exp(-dists_sq / (2 * sigma**2))

## y=Sigma alpha_i y_i K(x_i,x)+b
## dual problem: maximize W(alpha)=Sigma alpha_i - 0.5 Sigma alpha_i alpha_j y_i y_j K(x_i,x_j)
## s.t. 0<=alpha_i<=C, Sigma alpha_i y_i=0
## use smo algorithm to solve the dual problem
def smo(X,y,C,sigma,max_iter):
    m,n=X.shape
    alpha=np.zeros(m)
    b=0
    for iter in range(max_iter):
        for i in range(m):
            # calculate the decision value
            K=gaussian_kernel(X,X[i:i+1],sigma)
            f=np.sum(alpha*y*K)+b
            E=f-y[i]
            if (y[i]*E< -C and alpha[i]<C) or (y[i]*E>0 and alpha[i]>0):
                j=np.random.choice([x for x in range(m) if x!=i])
                K_j=gaussian_kernel(X,X[j:j+1],sigma)
                f_j=np.sum(alpha*y*K_j)+b
                E_j=f_j-y[j]
                alpha_i_old=alpha[i]
                alpha_j_old=alpha[j]
                if y[i]!=y[j]:
                    L=max(0,alpha[j]-alpha[i])
                    H=min(C,C+alpha[j]-alpha[i])
                else:
                    L=max(0,alpha[i]+alpha[j]-C)
                    H=min(C,alpha[i]+alpha[j])
                if L==H:
                    continue
                eta=2*K[0,j]-K[0,0]-K_j[0,0]
                if eta>=0:
                    continue
                alpha[j]=alpha[j]-y[j]*(E-E_j)/eta
                alpha[j]=min(H,alpha[j])
                alpha[j]=max(L,alpha[j])
                if abs(alpha[j]-alpha_j_old)<1e-5:
                    continue
                alpha[i]=alpha[i]+y[i]*y[j]*(alpha_j_old-alpha[j])
                b1=b-E-y[i]*(alpha[i]-alpha_i_old)*K[0,0]-y[j]*(alpha[j]-alpha_j_old)*K[0,j]
                b2=b-E_j-y[i]*(alpha[i]-alpha_i_old)*K[0,j]-y[j]*(alpha[j]-alpha_j_old)*K_j[0,0]
                if 0<alpha[i]<C:
                    b=b1
                elif 0<alpha[j]<C:
                    b=b2
                else:
                    b=(b1+b2)/2
    return alpha,b

def main():
    ## X.shape=(500,2),y.shape=(500,1)
    C=1.0
    sigma=1.0
    max_iter=1000
    alpha,b=smo(X,y,C,sigma,max_iter)
    K=gaussian_kernel(X,X,sigma)
    y_flatten=y.reshape(-1)
    y_pred=np.sign(K@(alpha*y_flatten)+b)
    # print("Accuracy:",np.mean(y_pred==y.flatten()))
    print(y_pred)

if __name__=="__main__":
    main()
